# Calidad del aire en Madrid: dióxido de nitrógeno (NO₂), 2019-2025

Análisis de los datos horarios de calidad del aire publicados por el Ayuntamiento de Madrid.

**Preguntas:**
1. ¿Cómo ha evolucionado el NO₂ entre 2019 y 2025?
2. ¿Qué estaciones registran niveles más altos y más bajos?
3. ¿Hay patrones por hora del día, día de la semana o estación del año?
4. ¿Cuántas veces se superan los valores límite (UE) y recomendados (OMS)?
5. *(Con prudencia)* ¿Se aprecia algún cambio durante el confinamiento de 2020?

**Fuente de datos:** Portal de Datos Abiertos del Ayuntamiento de Madrid
- [Calidad del aire. Datos horarios desde 2001](https://datos.madrid.es/dataset/201200-0-calidad-aire-horario)
- [Calidad del aire. Estaciones de control](https://datos.madrid.es/dataset/212629-0-estaciones-control-aire)

Licencia: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/legalcode.es).

## 1. Carga y limpieza de los datos

Las funciones están en `src/carga.py`. Aquí se ejecutan paso a paso para ver qué hace cada una
y cuántos datos se pierden en cada decisión.

Antes de ejecutar este cuaderno hay que descargar los datos con `python datos/descargar_datos.py`.

In [1]:
import sys
sys.path.append("..")  # para poder importar src/ desde la carpeta notebooks/

import pandas as pd
from src import carga

### 1.1 Lectura

Hasta 2024 hay un CSV por mes; desde 2025, un único CSV (que también trae 2026 y descartamos).
Cada fila del fichero original es **un día, en una estación, para un contaminante**, con 24 columnas
de valores (`H01`…`H24`) y 24 de códigos de validación (`V01`…`V24`).

In [2]:
datos = carga.leer_csv_horarios()
print("Filas leídas:", len(datos))
datos.head(3)

Filas leídas: 346032


,PROVINCIA,MUNICIPIO,ESTACION,MAGNITUD,PUNTO_MUESTREO,ANO,MES,DIA,H01,V01,...,H20,V20,H21,V21,H22,V22,H23,V23,H24,V24
0,28,079,4,1,28079004_1_38,2019,04,01,00004,V,...,00002,V,00002,V,00004,V,00004,V,00004,V
1,28,079,4,1,28079004_1_38,2019,04,02,00003,V,...,00003,V,00004,V,00004,V,00004,V,00004,V
2,28,079,4,1,28079004_1_38,2019,04,03,00003,V,...,00006,V,00005,V,00005,V,00005,V,00004,V


### 1.2 Solo NO₂

El NO₂ es la magnitud **8**, medida en µg/m³ (Anexo II del *Intérprete de ficheros* del Ayuntamiento).

In [3]:
no2 = carga.filtrar_contaminante(datos, magnitud=8)
print("Filas de NO2 (días × estación):", len(no2))

Filas de NO2 (días × estación): 60441


### 1.3 De formato ancho a formato largo

Pasamos de «una fila = un día con 24 horas» a «una fila = una hora». Este formato (*tidy data*)
es el que permite agrupar fácilmente por hora, día de la semana, mes o año.

In [4]:
largo = carga.pasar_a_formato_largo(no2)
print("Filas en formato largo:", len(largo), "| comprobación:", len(no2) * 24)
largo.head()

Filas en formato largo: 1450584 | comprobación: 1450584


,estacion,magnitud,fecha,hora,valor,validez
0,4,8,2019-04-01,1,17.0,V
1,4,8,2019-04-02,1,22.0,V
2,4,8,2019-04-03,1,33.0,V
3,4,8,2019-04-04,1,20.0,V
4,4,8,2019-04-05,1,22.0,V


### 1.4 Solo datos válidos

Cada hora lleva un código de validación: `V` (válido) o `N` (no válido). **Los datos `N` también
traen un número**, así que no basta con quitar celdas vacías: hay que filtrar por el código.
Según la documentación, los datos se anulan por mantenimiento, averías o valores anómalos.

In [5]:
print(largo["validez"].value_counts())
validos = carga.filtrar_validos(largo)

validez
V    1434945
N      15639
Name: count, dtype: int64
Filas totales: 1450584
Filas válidas: 1434945
Descartadas:   15639 (1.1%)


Horas descartadas por año:

In [6]:
resumen = (
    largo.assign(anio=largo["fecha"].dt.year, descartada=largo["validez"] != "V")
    .groupby("anio")["descartada"]
    .agg(total="size", descartadas="sum")
)
resumen["% descartadas"] = (100 * resumen["descartadas"] / resumen["total"]).round(2)
resumen

,total,descartadas,% descartadas
anio,,,
2019,208968,1291,0.62
2020,209088,1275,0.61
2021,207840,2361,1.14
2022,205464,1866,0.91
2023,207096,3263,1.58
2024,210696,2476,1.18
2025,201432,3107,1.54


### 1.5 Fecha y hora

Según la documentación, `H24` es la media de 23:00 a 24:00. Por tanto `Hk` es la media de la hora
*k-1* a la hora *k*, y **etiquetamos cada dato con el inicio de su intervalo** (`H01` → 00:00,
`H24` → 23:00 del mismo día).

Las horas están en **hora local de Madrid** (sin zona horaria), como las publica el Ayuntamiento.

In [7]:
con_fecha = carga.anadir_fecha_hora(validos)
con_fecha.sort_values(["estacion", "fecha_hora"])[["estacion", "fecha", "hora", "fecha_hora", "valor"]].head(3)

,estacion,fecha,hora,fecha_hora,valor
2180,4,2019-01-01,1,2019-01-01 00:00:00,60.0
62621,4,2019-01-01,2,2019-01-01 01:00:00,74.0
123062,4,2019-01-01,3,2019-01-01 02:00:00,80.0


### 1.6 Nombre y tipo de estación

Se cruza con el listado oficial de estaciones (como un BUSCARV en Excel). El tipo de estación
será importante: las de **tráfico** están junto a vías con mucha circulación; las de **fondo**,
lejos de fuentes directas; las **suburbanas**, en las afueras.

In [8]:
no2_limpio = carga.anadir_estaciones(con_fecha)
print("Filas sin nombre de estación:", no2_limpio["nombre_estacion"].isna().sum())
no2_limpio = no2_limpio[["fecha_hora", "estacion", "nombre_estacion", "tipo_estacion", "valor"]]
no2_limpio.head()

Filas sin nombre de estación: 0


,fecha_hora,estacion,nombre_estacion,tipo_estacion,valor
0,2019-04-01,4,Plaza de España,Urbana tráfico,17.0
1,2019-04-02,4,Plaza de España,Urbana tráfico,22.0
2,2019-04-03,4,Plaza de España,Urbana tráfico,33.0
3,2019-04-04,4,Plaza de España,Urbana tráfico,20.0
4,2019-04-05,4,Plaza de España,Urbana tráfico,22.0


### 1.7 Comprobaciones

In [9]:
print("Filas finales:", len(no2_limpio))
print("Duplicados (misma estación y hora):", no2_limpio.duplicated(["estacion", "fecha_hora"]).sum())
print("Valores negativos:", (no2_limpio["valor"] < 0).sum())
print("Desde", no2_limpio["fecha_hora"].min(), "hasta", no2_limpio["fecha_hora"].max())
no2_limpio["valor"].describe().round(1)

Filas finales: 1434945
Duplicados (misma estación y hora): 0
Valores negativos: 0
Desde 2019-01-01 00:00:00 hasta 2025-12-31 23:00:00


count    1434945.0
mean          27.5
std           23.5
min            0.0
25%           11.0
50%           20.0
75%           38.0
max          328.0
Name: valor, dtype: float64

**Cobertura:** porcentaje de horas con dato válido sobre las horas del año, por estación.
Un 0 % significa que no hay ningún dato válido ese año.

In [10]:
horas_anio = no2_limpio.groupby(no2_limpio["fecha_hora"].dt.year)["fecha_hora"].nunique()
cobertura = (
    no2_limpio.groupby(["nombre_estacion", no2_limpio["fecha_hora"].dt.year])
    .size()
    .unstack(fill_value=0)
    .div(horas_anio)
    .mul(100)
    .round(1)
)
cobertura.columns.name = "año"
cobertura

año,2019,2020,2021,2022,2023,2024,2025
nombre_estacion,,,,,,,
Arturo Soria,99.7,99.4,99.6,71.9,99.3,99.1,98.9
Barajas Pueblo,99.8,99.8,99.5,99.2,99.0,98.9,98.2
Barrio del Pilar,99.3,98.7,99.6,98.9,98.5,98.3,98.5
Casa de Campo,99.2,99.4,95.6,98.7,94.0,97.7,97.6
Castellana,99.8,99.8,99.8,99.3,99.3,99.3,98.8
Cuatro Caminos,95.7,99.6,98.9,99.3,98.9,98.6,98.3
El Pardo,99.7,99.7,99.7,99.5,98.7,98.1,98.0
Ensanche de Vallecas,99.9,99.6,99.5,99.1,99.0,99.1,98.8
Escuelas Aguirre,95.2,98.5,97.2,99.2,99.0,98.7,98.4


Estaciones y años con **menos del 90 % de horas con dato válido**. Hay que tenerlos en cuenta
al comparar medias anuales entre estaciones (fase 3): una media calculada con muchas horas
ausentes puede no ser representativa del año completo.

In [11]:
baja_cobertura = cobertura.stack()
baja_cobertura[baja_cobertura < 90].rename("% horas válidas").to_frame()

% horas válidas
nombre_estacion  año                  
Arturo Soria     2022             71.9
Plaza de España  2020             77.3
                 2021             71.1
                 2022             75.7
Plaza del Carmen 2023             77.2
Ramón y Cajal    2023             83.7
Urb. Embajada    2025              0.0

### 1.8 Decisiones de limpieza (resumen)

| Decisión | Motivo |
|---|---|
| Periodo 2019-2025 (años completos) | 2019 como referencia previa a 2020; 2026 aún incompleto. |
| Solo NO₂ (magnitud 8) | Es el contaminante de las preguntas. |
| Solo horas con código `V` | Los datos `N` fueron anulados en la validación oficial aunque traigan número. |
| `Hk` → inicio del intervalo (k-1):00 | `H24` es la media de 23:00 a 24:00 según la documentación. |
| Hora local, sin zona horaria | Así se publica. Los días de cambio de hora tienen 23 o 25 horas (el Ayuntamiento elimina la hora repetida). |
| Se conservan las estaciones con baja cobertura | Se marcan en la tabla anterior; cómo tratarlas se decide en el análisis. |

**Limitación conocida:** la estación *Urb. Embajada* (55) no tiene ningún dato publicado en 2025,
aunque el listado oficial de estaciones indica que mide NO₂. Los datos no permiten saber el motivo.

*Los porcentajes y recuentos concretos son los que muestran las celdas de arriba.*